In [2]:
import pandas as pd
import re
import numpy as np
import multiprocessing
import string

from typing import List

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

# Constants and configuration
CORES = multiprocessing.cpu_count()
STOPWORDS = list(stopwords.words('english')) + [
    "'d", "'ll", "'re", "'s", "'ve", '``', 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would'
]

def tokenize_lemmatize_remove_stop_words(text: str) -> List[str]:
    return [wnl.lemmatize(token) for token in word_tokenize(text) if 
            token not in STOPWORDS and
            len(token) > 2 and  # Mots de moins de 2 lettres
            not (bool(re.search(r'\d', token))) and # Mots contenant des chiffres
            not (any(char in string.punctuation for char in token)) # Mots contenant des signes de ponctuation
]

data = pd.read_excel(
    '../1-data/training_datasets/train_dataset_30pc.xlsx')

dic = {'neutral':0, 'incel': 1}
data['label'] = data['category'].map(dic)

vectorizer = TfidfVectorizer(
    stop_words=STOPWORDS,
    token_pattern=None,
    tokenizer=tokenize_lemmatize_remove_stop_words,
    max_features=15000
)

X = vectorizer.fit_transform(data['text_post'].astype('str'))
y = data['label']

feature_names = vectorizer.get_feature_names_out()

model = LogisticRegression(
    n_jobs=CORES,
)

model.fit(X, y)

LogisticRegression(n_jobs=6)

In [7]:
coefficients = model.coef_[0]

feature_importance = pd.DataFrame({'Trait': feature_names, 'Coefficient': coefficients})
feature_importance = feature_importance.sort_values('Coefficient', ascending=False)[:250]
feature_importance['Trait'].tolist()

['incel',
 'chad',
 'woman',
 'ugly',
 'incels',
 'normies',
 'alone',
 'virgin',
 'loneliness',
 'relationship',
 'attractive',
 'normie',
 'social',
 'life',
 'cope',
 'personality',
 'dating',
 'girl',
 'hobby',
 'lonely',
 'someone',
 'gaming',
 'tinder',
 'conversation',
 'suicide',
 'anime',
 'height',
 'blackpill',
 'rejection',
 'interest',
 'depressed',
 'nerdy',
 'nerd',
 'jfl',
 'loser',
 'date',
 'men',
 'girlfriend',
 'female',
 'average',
 'manlet',
 'people',
 'gamer',
 'cuck',
 'single',
 'introvert',
 'fa',
 'relate',
 'attracted',
 'sub',
 'rope',
 'forever',
 'talk',
 'socially',
 'happiness',
 'shower',
 'awkward',
 'subhuman',
 'genetics',
 'tall',
 'existence',
 'person',
 'elaborate',
 'gym',
 'soy',
 'confidence',
 'foids',
 'enjoy',
 'shy',
 'rejected',
 'ldar',
 'surgery',
 'confident',
 'luck',
 'unattractive',
 'self',
 'friend',
 'bro',
 'physical',
 'bluepilled',
 'success',
 'music',
 'mogs',
 'whore',
 'nothing',
 'study',
 'depressing',
 'studying',
 'b

In [4]:
# Create LaTeX table string
latex_table = "\\begin{table}[htbp]\n\\centering\n\\begin{tabular}{|c|c|}\n\\hline\nTrait & Coefficient \\\\\n\\hline\n"
for row in feature_importance.to_dict('records'):
    latex_table += f"{row['Trait']} & {row['Coefficient']:.4f} \\\\\n"
latex_table += "\\hline\n\\end{tabular}\n\\caption{25 traits les plus prédicitfs de la classe 'incels'}\n\\label{tab:top_features}\n\\end{table}"

print(latex_table)

\begin{table}[htbp]
\centering
\begin{tabular}{|c|c|}
\hline
Trait & Coefficient \\
\hline
incel & 6.5982 \\
chad & 6.5319 \\
woman & 5.7303 \\
ugly & 5.7011 \\
incels & 5.5302 \\
normies & 4.6894 \\
alone & 4.6703 \\
virgin & 4.5697 \\
loneliness & 4.5553 \\
relationship & 4.4181 \\
attractive & 4.3775 \\
normie & 4.2995 \\
social & 4.2354 \\
life & 4.2053 \\
cope & 4.0723 \\
personality & 4.0205 \\
dating & 3.9777 \\
girl & 3.9139 \\
hobby & 3.8762 \\
lonely & 3.7593 \\
someone & 3.6457 \\
gaming & 3.4611 \\
tinder & 3.4367 \\
conversation & 3.1805 \\
suicide & 3.1409 \\
anime & 3.1263 \\
height & 3.0915 \\
blackpill & 3.0891 \\
rejection & 3.0537 \\
interest & 3.0062 \\
depressed & 3.0019 \\
nerdy & 2.9354 \\
nerd & 2.9185 \\
jfl & 2.9147 \\
loser & 2.8895 \\
date & 2.8239 \\
men & 2.7422 \\
girlfriend & 2.6688 \\
female & 2.6166 \\
average & 2.5154 \\
manlet & 2.4798 \\
people & 2.4642 \\
gamer & 2.4432 \\
cuck & 2.4359 \\
single & 2.4016 \\
introvert & 2.3639 \\
fa & 2.3395 \\
rel